<a href="https://colab.research.google.com/github/yashhotey/RAG_Implementation/blob/main/RAG_Implemented_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#importing the required libraries
!pip install langchain langchain-google-genai langchain-community langchainhub python-dotenv langchain-Chroma beautifulsoup4 pypdf

  Using cached langchain_google_genai-4.2.5-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchainhub-0.1.21-py3-none-any.whl.metadata (659 bytes)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.0/349.0 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.5 MB/s et

In [1]:
#creating a .env file to keep api key secure
env_content="""
GEMINI_API_KEY="your_api_key"
"""
with open(".env","w") as file:
  file.write(env_content)
print("file created successfully")

file created successfully


In [2]:
#setting up the model
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()

llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash",google_api_key=os.getenv("GEMINI_API_KEY"))

In [8]:
#taking multiple types of files and urls as input and segregating to functions accordingly
import os

from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader, CSVLoader, TextLoader

def sort_input(source: str) -> list[Document]:
    if source.startswith(("http://","https://")):
        loader=WebBaseLoader(source)
        return loader.load()
    else:
      return load_uploaded_file(source)

def load_uploaded_file(file_path: str) -> list[Document]:
  _,extension=os.path.splitext(file_path) #taking only the extension and storing from created tuple
  extension=extension.lower()
  if extension == ".pdf":
    loader=PyPDFLoader(file_path)
  elif extension == ".csv":
    loader=CSVLoader(file_path)
  elif extension == ".txt":
    loader=TextLoader(file_path)
  else:
    raise ValueError(f"unsupported file type: {extension}")

  return loader.load()

In [10]:
from google.colab import files

# This will prompt you to select a file from your computer
uploaded = files.upload()

# Get the exact filename of the uploaded file
uploaded_filename = list(uploaded.keys())[0]
print(f"\nSuccessfully uploaded: {uploaded_filename}")


Saving APA style of referencing.pdf to APA style of referencing.pdf

Successfully uploaded: APA style of referencing.pdf


In [11]:
data=sort_input(uploaded_filename)

In [12]:
#splitting the document using text_splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

textsplitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
splitted=textsplitter.split_documents(data)

In [13]:
print(len(splitted))

10


In [14]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
import os
from dotenv import load_dotenv
load_dotenv()

embedding_model=GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview",
    batch_size=10,
    google_api_key=os.getenv("GEMINI_API_KEY")
    )

vectorstore=Chroma.from_documents(
    documents=splitted,
    embedding=embedding_model
)

In [15]:
print(vectorstore._collection.count())

10


In [16]:
print(vectorstore._collection.get())

{'ids': ['907d6ad1-1821-49dd-ae0a-52a73c1166d8', 'fa6b1014-9d7e-4e34-88e5-7e5c0151d6ff', '22773e80-7e8e-4ada-b309-c75543a92861', 'dfd74a59-434e-4177-8af0-0bac2ee660ab', '444b966a-371b-4f55-8faa-93807e23d6a8', 'ad8806fe-c81d-45e4-8483-81267a4a8a3d', '81dcfbfa-89e6-499f-8767-c67484904125', 'e0d80b34-2f37-4285-94e6-5e13fa61249f', '80223e0d-e395-4099-9a1b-b998f4459994', '585d3fc7-fb93-4894-8257-84a9c9e90e8e'], 'embeddings': None, 'documents': ['7th edition\nCommon Reference Examples Guide\nThis guide contains examples of common types of APA Style references. Section numbers indicate where \nto find the examples in the Publication Manual of the American Psychological Association (7th ed.). \nMore information on references and reference examples are in Chapters 9 and 10 of the Publication \nManual as well as the Concise Guide to APA Style (7th ed.). Also see the Reference Examples pages on \nthe APA Style website.\nJournal Article (Section 10.1)\nEdwards, A. A., Steacy, L. M., Siegelman, N.,

In [17]:
retriever=vectorstore.as_retriever()

In [18]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n\nContext: {context}"),
    ("human", "{question}"),
])

In [19]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [20]:
def formatdocs(docs):
  return "\n".join(doc.page_content for doc in docs)

In [21]:
rag_chain=({"context":retriever | formatdocs,"question":RunnablePassthrough()}
           | prompt
           | llm
           | StrOutputParser())

In [22]:
rag_chain.invoke("what is the document about?")

'This document is a guide to common APA Style reference examples. It provides examples for various types of sources like journal articles, press releases, conference sessions, dissertations, and more, along with their corresponding section numbers in the Publication Manual of the American Psychological Association (7th ed.).'

In [ ]:
rag_chain.invoke("What does the page try to explain in Namespace Considerations?")

'The "Namespace considerations" page explains how different types of pages are organized on Wikipedia using namespaces. It clarifies that only encyclopedia articles are created without a namespace prefix, while all other pages are prefixed by their respective namespace followed by a colon. The page also provides a comprehensive list of these various namespaces.'

In [23]:
rag_chain.invoke("Can you give me one exapmle from the document?")

'One example from the document is a journal article:\n\nEdwards, A. A., Steacy, L. M., Siegelman, N., Rigobon, V. M., Kearns, D. M., Rueckl, J. G., & Compton, D. L. (2022). Unpacking the unique relationship between set for variability and word reading development: Examining word- and child-level predictors of performance. *Journal of Educational Psychology*, *114*(6), 1242–1256. https://doi.org/10.1037/edu0000696'

In [24]:
rag_chain.invoke("Can you create an exapmle not in the document?")

'Here is an example for an online newspaper article, which is not included in the provided document:\n\nDoe, J. (2023, October 26). The future of AI in everyday life. *The Daily Chronicle*. https://www.dailychronicle.com/ai-future-article'